In [ ]:
# CAPM-Based Portfolio Optimization and Performance Analysis

## Overview
This project applies portfolio optimization techniques to 10 industry return series.
Using a mean-variance framework, it compares constrained and unconstrained portfolios and evaluates their risk-adjusted performance using return, 
volatility, Sharpe ratio, and cumulative wealth paths.

In [ ]:
## Data Processing

The dataset contains return series for 10 industries.
In this section, I load the data, convert returns into decimal form, and estimate the sample mean vector and covariance matrix, which are the
main inputs for portfolio optimization.

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import minimize

PATH_INDU10 = ...
R = pd.read_excel(PATH_INDU10)
...

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import minimize

PATH_INDU10 = r"C:\Users\gavin\Downloads\Python2\Indu10_July26_July11.xlsx"  
R = pd.read_excel(PATH_INDU10) 
R = R.filter(like='Indu').div(100).dropna()  
mu = R.mean().values      
Sigma = R.cov().values
n, gamma = R.shape[1], 3         
ones = np.ones(n)

def port_stats(w):       
    m = w @ mu
    s = np.sqrt(w @ Sigma @ w)
    return m, s, m/s

In [ ]:
1)   What are the alphas of the CAPM regression ?

In [ ]:
def obj(w): return -(w @ mu - 0.5*gamma * (w @ Sigma @ w))
cons = ({'type':'eq','fun': lambda w: np.sum(w)-1}) 
bnds = [(0.0, 0.3)] * n  
w_con = minimize(obj, x0=np.repeat(1/n,n), bounds=bnds, constraints=cons).x

In [ ]:
2)  Which are the two largest alphas in terms of absolute values ?

In [ ]:
mu_con, sd_con, sr_con = port_stats(w_con)

In [ ]:
4)   Is the CAPM rejected by the data or not?

In [ ]:
bnds_nc = [(None, None)] * n
w_unc = minimize(obj, x0=np.repeat(1/n,n), bounds=bnds_nc, constraints=cons).x
mu_unc, sd_unc, sr_unc = port_stats(w_unc)

In [ ]:
## Principal Component Analysis (PCA)

This section analyzes the covariance structure of industry returns using PCA and evaluates its implications for portfolio construction.

In [ ]:
w_mkt = np.repeat(1/n, n)
mu_mkt, sd_mkt, sr_mkt = port_stats(w_mkt)

print("Constrained weights (<= 0.3):", np.round(w_con, 4))
print("Constrained: mean/std/sharpe =", round(mu_con, 6), round(sd_con, 6), round(sr_con, 6))
print("Unconstrained: mean/std/sharpe =", round(mu_unc, 6), round(sd_unc, 6), round(sr_unc, 6))
print("Market: mean/std/sharpe =", round(mu_mkt, 6), round(sd_mkt, 6), round(sr_mkt, 6))

In [ ]:
1)   What are the eigenvalues and eigenvectors of the sample covariance matrix ?

In [ ]:
def wealth_path(w, R_):
    ret = R_.values @ w
    return pd.Series((1 + ret).cumprod(), index=R_.index)

W_con = wealth_path(w_con, R)
W_unc = wealth_path(w_unc, R)
W_mkt = wealth_path(w_mkt, R)

print("Final wealth (Constrained, Unconstrained, Market):",
      round(W_con.iloc[-1], 4),
      round(W_unc.iloc[-1], 4),
      round(W_mkt.iloc[-1], 4))

In [ ]:
2)  Which are the coefficients for the first PCA?

In [ ]:
import numpy as np
x = 0.10 + 0.20*np.random.randn(60)
print(x.mean(), x.var(ddof=1))


In [ ]:
3)   What is the R-squared of the regression of the industry returns on the first PCA?

In [ ]:
for T in [600,6000]:
    x = 0.10 + 0.20*np.random.randn(T)
    print(T, x.mean(), x.var(ddof=1))


In [ ]:
4)   In comparison with regression on the market index, is the PCA better or worse in      terms of the value of the R-squared ?

In [ ]:
from scipy.stats import norm
S0=K=100; r=0.05; s=0.2; T=1
d1=(np.log(S0/K)+(r+0.5*s*s)*T)/(s*np.sqrt(T)); d2=d1-s*np.sqrt(T)
bs=S0*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
for M in [1000,1_000_000]:
    Z=np.random.randn(M); ST=S0*np.exp((r-0.5*s*s)*T+s*np.sqrt(T)*Z)
    print(M, np.exp(-r*T)*np.maximum(ST-K,0).mean(), bs)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(W_con, label="Constrained")
plt.plot(W_unc, label="Unconstrained")
plt.plot(W_mkt, label="Market")
plt.legend()
plt.title("Portfolio Performance")
plt.show()

In [ ]:
## Conclusion

The unconstrained portfolio achieves higher Sharpe ratio, while constraints reduce risk but limit performance.